# Lab 03 — Batch Size Benchmark
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chelsietao/ai-inference-lab/blob/main/notebooks/03_batch_size_benchmark.ipynb)

> 觀察 batch size 如何改變 latency 與 throughput,並理解兩者的取捨。

## 學習目標
1. 區分 **per-batch latency** 與 **per-image latency**。
2. 理解 batch 越大 throughput 不一定線性提升(parallelism、GPU utilization、記憶體限制)。
3. 學會處理 Out-of-Memory:程式會**記錄**失敗並停止,而不是崩潰。

## 先備知識
Lab 02。

## 需要 GPU 嗎?
不需要,CPU 即可(`DEVICE = "cpu"`)。`"auto"` 會優先使用 CUDA,其次 Apple MPS,最後 CPU。⚠️ GPU 路徑**尚未在本機以 CUDA 驗證**,請在 Colab GPU runtime 自行確認。

## 技術原理
- **Parallelism**:較大 batch 提供更多可平行處理的工作,硬體利用率提高。
- **GPU Utilization**:batch=1 時 GPU 多數運算單元閒置;batch 變大才逐漸填滿。
- **Memory constraint**:activations 與 input 隨 batch 線性增加,過大會 OOM。Colab Free 的 GPU 記憶體有限,且不保證型號。
- **Latency vs Throughput trade-off**:batch 越大,單一批次(也就是排在這批的每個請求)等待越久,但單位時間處理的影像數可能更高。

## 1. 參數

In [ ]:
DEVICE = "auto"                   # auto 的優先順序:CUDA > MPS > CPU(Colab 上只有 CUDA 或 CPU)
PRECISION = "fp32"                # CUDA 時可選 "fp16"
BATCH_SIZES = [1, 4, 8, 16, 32]
WARMUP_ITERS = 5
MEASURE_ITERS = 20
USE_PRETRAINED = True

In [ ]:
# 安裝 AI Inference Lab 套件(只在尚未安裝時執行;PyTorch / torchvision 在 Colab 已內建)
import importlib.util, subprocess, sys
if importlib.util.find_spec("ai_inference_lab") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/chelsietao/ai-inference-lab.git@main"])

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import ai_inference_lab as lab
print("ai_inference_lab", lab.__version__)

In [ ]:
hw = lab.detect_hardware(DEVICE)
print(hw.summary())

In [ ]:
loaded = lab.load_classifier(pretrained=USE_PRETRAINED)
run_device = lab.select_device(DEVICE).type   # 實際使用的 device(不可用時會退回 CPU)
print("實際使用的 device:", run_device)

## 2. 執行 Batch Size Sweep
如果某個 batch size 發生 OOM,會被記錄在 `errors` 欄位並停止後續更大的 batch。

In [ ]:
records = lab.sweep_batch_sizes(loaded, run_device, PRECISION, BATCH_SIZES, WARMUP_ITERS, MEASURE_ITERS)
lab.summary_table(records)

## 3. 視覺化
注意圖 1 的兩條線:**per batch** 是一整批的時間,**per image** 是平均每張的時間——它們不是同一件事。

In [ ]:
import matplotlib.pyplot as plt
from ai_inference_lab.visualization import charts
charts.plot_batch_latency(records, "results/batch_latency.png")
charts.plot_batch_throughput(records, "results/batch_throughput.png")
charts.plot_memory(records, "results/batch_memory.png")
plt.show()

## 4. 匯出結果

In [ ]:
lab.save_json(records, "results/batch_size.json")
lab.save_csv(records, "results/batch_size.csv")

## 預期觀察(不預先寫數字)
- **Per-batch latency** 隨 batch size 增加而上升。
- **Per-image latency** 與 **throughput** 可能改善後趨於持平(飽和)——在哪個 batch size 飽和,取決於你的硬體,請觀察你自己的圖。
- CPU 上改善幅度通常小於 GPU;核心數越少越早飽和。
- 在 CUDA 上,peak memory 應隨 batch size 增加。

## 常見錯誤
- 把 per-batch latency 當成 per-image latency。
- 假設所有 batch size 都能在 free-tier GPU 上執行。
- 只看 throughput,忽略單一請求的等待時間(線上服務通常受 latency SLA 限制)。

## 真實產業應用
Dynamic batching(例如 NVIDIA Triton Inference Server)會把短時間內到達的請求合併成 batch,在 latency 限制內提高 throughput。

## 相關概念
Dynamic batching、Throughput vs Latency、GPU utilization、Memory-bound vs Compute-bound。

## 官方參考資料
- [PyTorch CUDA memory management](https://pytorch.org/docs/stable/notes/cuda.html#memory-management)
- [NVIDIA Triton dynamic batching](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_configuration.html#dynamic-batcher)